## RAG Application Using Type Sense

In [1]:
import typesense
import json

client = typesense.Client({
    'nodes': [{
        'host': 'si86yd5pjzu039awp-1.a1.typesense.net',
        'port': '443',
        'protocol': 'https'
    }],
    'api_key': '2QvebgEDLrO2qikS0vTmtxu5vWLzW3Us',
    'connection_timeout_seconds': 2
})

# ── Schema ──────────────────────────────────────────────────────────────────
books_schema = {
    'name': 'books',
    'fields': [
        {'name': 'title',            'type': 'string'},
        {'name': 'authors',          'type': 'string[]', 'facet': True},
        {'name': 'publication_year', 'type': 'int32',    'facet': True},
        {'name': 'ratings_count',    'type': 'int32'},
        {'name': 'average_rating',   'type': 'float'},
        {'name': 'image_url',        'type': 'string',   'optional': True},  # ✅ added missing field
    ],
    'default_sorting_field': 'ratings_count'
}

# Drop collection if it already exists, then recreate
try:
    client.collections['books'].delete()
    print("Old collection deleted.")
except Exception:
    pass

print(client.collections.create(books_schema))

# ── Fix id field + Import ────────────────────────────────────────────────────
fixed_lines = []
with open('books.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        doc = json.loads(line.strip())
        doc['id'] = str(doc['id'])   # ✅ convert int id → string (Typesense requirement)
        fixed_lines.append(json.dumps(doc))

jsonl_data = '\n'.join(fixed_lines)

response = client.collections['books'].documents.import_(jsonl_data, {'action': 'create'})

# ── Check results ────────────────────────────────────────────────────────────
results = [json.loads(r) for r in response.split('\n') if r]
failed  = [r for r in results if not r.get('success')]

print(f"✅ Imported: {len(results) - len(failed)} / {len(results)}")
if failed:
    print(f"❌ Failed: {len(failed)}")
    for f in failed[:5]:   # show first 5 failures
        print(f)

# Confirm final count
print(client.collections['books'].retrieve())

Old collection deleted.
{'created_at': 1772864759, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'f

In [2]:
client

In [3]:
with open('books.jsonl', 'r', encoding = 'utf-8') as jsonl_file :
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [4]:
search_parameters = {
    'q' : "Nice text",
    'query_by' : "title, authors",
    'sort_by' : "ratings_count : desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['Joseph Bond', 'Tracy Jones'],
    'average_rating': 3.31,
    'id': '3',
    'image_url': 'https://example.com/book_covers/book_3.jpg',
    'publication_year': 1969,
    'ratings_count': 26581,
    'title': 'Nice next'},
   'highlight': {'title': {'matched_tokens': ['Nice', 'next'],
     'snippet': '<mark>Nice</mark> <mark>next</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Nice', 'next'],
     'snippet': '<mark>Nice</mark> <mark>next</mark>'}],
   'text_match': 1157451402730014841,
   'text_match_info': {'best_field_score': '2211864317953',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451402730014841',
    'tokens_matched': 2,
    'typo_prefix_score': 2}}],
 'out_of': 1200,
 'page': 1,
 'request_params': {'collection_name': 'books',
  'first_q': 'nice',
  'per_page': 10,
  'q': 'Nice text'},
 'search_cutoff': False,
 'search_time_ms':

In [5]:
search_paramters = {
    'q'         : 'Nice text',
    'query_by'  : 'title',
    'filter_by' : 'publication_year : < 1998',
    'sort_by'   : 'publication_year : desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['Joseph Bond', 'Tracy Jones'],
    'average_rating': 3.31,
    'id': '3',
    'image_url': 'https://example.com/book_covers/book_3.jpg',
    'publication_year': 1969,
    'ratings_count': 26581,
    'title': 'Nice next'},
   'highlight': {'title': {'matched_tokens': ['Nice', 'next'],
     'snippet': '<mark>Nice</mark> <mark>next</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Nice', 'next'],
     'snippet': '<mark>Nice</mark> <mark>next</mark>'}],
   'text_match': 1157451402730014841,
   'text_match_info': {'best_field_score': '2211864317953',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451402730014841',
    'tokens_matched': 2,
    'typo_prefix_score': 2}}],
 'out_of': 1200,
 'page': 1,
 'request_params': {'collection_name': 'books',
  'first_q': 'nice',
  'per_page': 10,
  'q': 'Nice text'},
 'search_cutoff': False,
 'search_time_ms':

In [6]:
search_parameters = {
    'q'     : 'Note section',
    'query_by'  : 'title',
    'facet_by' : 'authors',
    'sort_by' : 'average_rating : desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [{'counts': [{'count': 1,
     'highlighted': 'Jose Owens',
     'value': 'Jose Owens'},
    {'count': 1, 'highlighted': 'Michelle Paul', 'value': 'Michelle Paul'},
    {'count': 1,
     'highlighted': 'Brandon Coleman',
     'value': 'Brandon Coleman'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 3}}],
 'found': 1,
 'hits': [{'document': {'authors': ['Brandon Coleman',
     'Jose Owens',
     'Michelle Paul'],
    'average_rating': 2.97,
    'id': '1',
    'image_url': 'https://example.com/book_covers/book_1.jpg',
    'publication_year': 1969,
    'ratings_count': 36577,
    'title': 'Note section'},
   'highlight': {'title': {'matched_tokens': ['Note', 'section'],
     'snippet': '<mark>Note</mark> <mark>section</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Note', 'section'],
     'snippet': '<mark>Note</mark> <mark>section</mark>'}],
   'text_match': 1157451471449491577,
   'text_match_info': {'best_field

In [7]:
### Langchain + Typesense + Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq



d:\DAK_AIML_Resume Projects\RAG Pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

In [10]:
docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params = {
        "host" : "si86yd5pjzu039awp-1.a1.typesense.net", # Use xxx.a1.typesense.net for Typesense Cloud
        "port" : "443", # Use 443 for typesense cloud
        "protocol" : "https", # Use https for typesense Cloud
        "typesense_api_key" : "2QvebgEDLrO2qikS0vTmtxu5vWLzW3Us",
        "typesense_collection_name" : "lang-chain"
    },

)

In [11]:
query = "What is Artificial Intelligence"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Artificial Intelligence: A Comprehensive Overview

Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, problem-solving, perception, and language understanding. AI was formally founded as an academic discipline in 1956 at the Dartmouth Conference, where researchers John McCarthy, Marvin Minsky, Nathaniel Rochester, and Claude Shannon proposed that every aspect of learning and every feature of intelligence could be so precisely described that a machine could be made to simulate it. Since then, AI has gone through several phases of optimism, funding, and disillusionment â€” periods known as "AI winters" â€” before experiencing the explosive growth we see today driven by massive datasets and powerful computing hardware.

Machine Learning: The Core of Modern AI


In [12]:
### Retriever 

retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000001F780082990>, search_kwargs={})

In [13]:
query = "Artificial Intelligence In-depth Explanation"
retriever.invoke(query)[0]

Document(metadata={'source': 'test.txt'}, page_content='Artificial Intelligence: A Comprehensive Overview\n\nArtificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, problem-solving, perception, and language understanding. AI was formally founded as an academic discipline in 1956 at the Dartmouth Conference, where researchers John McCarthy, Marvin Minsky, Nathaniel Rochester, and Claude Shannon proposed that every aspect of learning and every feature of intelligence could be so precisely described that a machine could be made to simulate it. Since then, AI has gone through several phases of optimism, funding, and disillusionment â€” periods known as "AI winters" â€” before experiencing the explosive growth we see today driven by massive datasets and powerful computing hardware.\n\nMachine Learning: The Core of Modern AI')